In [20]:
#### DROPOUT RATES FOR NON-RURAL SCHOOL

import pandas as pd

grad_path = "/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/grad_nonrural.csv"
grad_df = pd.read_csv(grad_path, dtype=str)
grad_df.columns = grad_df.columns.str.strip()

# Replace "-" with NaN first
grad_df["NUM_DROPOUT"] = grad_df["NUM_DROPOUT"].replace("-", float('nan'))
grad_df["TOTAL_ENROLLED"] = pd.to_numeric(grad_df["TOTAL_ENROLLED"], errors="coerce")

# Convert to numeric
grad_df["NUM_DROPOUT"] = pd.to_numeric(grad_df["NUM_DROPOUT"], errors="coerce")

# Group by SUBGROUP_NAME and calculate dropout rate
subgroup_stats = (
    grad_df.dropna(subset=["TOTAL_ENROLLED"])
    .groupby("SUBGROUP_NAME", as_index=False)
    .agg({
        "NUM_DROPOUT": "sum",
        "TOTAL_ENROLLED": "sum",
    })
)

# Calculate dropout rate for each subgroup
subgroup_stats["DROPOUT_RATE"] = (
    (subgroup_stats["NUM_DROPOUT"] / subgroup_stats["TOTAL_ENROLLED"]) * 100
).round(2)

# Replace inf/nan with 0
subgroup_stats["DROPOUT_RATE"] = subgroup_stats["DROPOUT_RATE"].replace([float('inf'), float('-inf')], 0)

print(subgroup_stats)

                    SUBGROUP_NAME  NUM_DROPOUT  TOTAL_ENROLLED  DROPOUT_RATE
0                    All Students       6855.0          169318          4.05
1        English Language Learner       1706.0           11700         14.58
2   Ever-English Language Learner        585.0           27595          2.12
3  Never-English Language Learner       4528.0          130023          3.48
4    Non-English Language Learner       4427.0          157618          2.81


In [21]:
#### CREATING A NEW CLASSIFICATION OF RURAL IN NEW CSV

import pandas as pd

in_path = "/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/grad.csv"
out_path = "/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/grad2.csv"

remove_entities = {
    "NEWBURGH CITY SD",
    "UTICA CITY SD",
    "MIDDLETOWN CITY SD",
    "MONROE-WOODBURY CSD",
    "KINGSTON CITY SD",
    "SCHENECTADY CITY SD",
    "FALLSBURG CSD",
    "LIBERTY CSD",
    "AMSTERDAM CITY SD",
    "ITHACA CITY SD",
    "BINGHAMTON CITY SD",
    "MONTICELLO CSD",
    "WASHINGTONVILLE CSD",
    "JOHNSON CITY CSD",
    "KIRYAS JOEL VILLAGE UFSD",
    "ROTTERDAM-MOHONASEN CSD",
    "NISKAYUNA CSD",
    "UTICA ACADEMY OF SCIENCE CS",
    "GOSHEN CSD",
}

df = pd.read_csv(in_path, dtype=str)
df.columns = df.columns.str.strip()
df["ENTITY_NAME"] = df["ENTITY_NAME"].fillna("").astype(str).str.strip()

grad2 = df[~df["ENTITY_NAME"].isin(remove_entities)].copy()
grad2.to_csv(out_path, index=False)

print("Input rows:", len(df))
print("Output rows:", len(grad2))
print("Removed rows:", len(df) - len(grad2))
print("Saved:", out_path)

Input rows: 1390
Output rows: 1295
Removed rows: 95
Saved: /Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/grad2.csv


In [23]:
#### CREATING A NEW CLASSIFICATION OF NON-RURAL IN NEW CSV

import pandas as pd

grad_path = "/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/grad.csv"
nonrural_path = "/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/grad_nonrural.csv"
out_path = "/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/grad2_nonrural.csv"

schools_to_add = {
    "NEWBURGH CITY SD",
    "UTICA CITY SD",
    "MIDDLETOWN CITY SD",
    "MONROE-WOODBURY CSD",
    "KINGSTON CITY SD",
    "SCHENECTADY CITY SD",
    "FALLSBURG CSD",
    "LIBERTY CSD",
    "AMSTERDAM CITY SD",
    "ITHACA CITY SD",
    "BINGHAMTON CITY SD",
    "MONTICELLO CSD",
    "WASHINGTONVILLE CSD",
    "JOHNSON CITY CSD",
    "KIRYAS JOEL VILLAGE UFSD",
    "ROTTERDAM-MOHONASEN CSD",
    "NISKAYUNA CSD",
    "UTICA ACADEMY OF SCIENCE CS",
    "GOSHEN CSD",
}

# Load files
grad = pd.read_csv(grad_path, dtype=str)
nonrural = pd.read_csv(nonrural_path, dtype=str)

# Clean column names / entity names
grad.columns = grad.columns.str.strip()
nonrural.columns = nonrural.columns.str.strip()
grad["ENTITY_NAME"] = grad["ENTITY_NAME"].fillna("").astype(str).str.strip().str.upper()
nonrural["ENTITY_NAME"] = nonrural["ENTITY_NAME"].fillna("").astype(str).str.strip().str.upper()

# Pull all rows ("stats") for requested schools from grad.csv
to_add = grad[grad["ENTITY_NAME"].isin(schools_to_add)].copy()

# Align columns to nonrural file, then append
to_add = to_add[[c for c in nonrural.columns if c in to_add.columns]]
result = pd.concat([nonrural, to_add], ignore_index=True)

# Remove exact duplicate rows if any
result = result.drop_duplicates()

# Save
result.to_csv(out_path, index=False)

print("Rows in grad_nonrural.csv:", len(nonrural))
print("Rows pulled from grad.csv:", len(to_add))
print("Rows in grad2_nonrural.csv:", len(result))
print("Saved:", out_path)

Rows in grad_nonrural.csv: 2495
Rows pulled from grad.csv: 95
Rows in grad2_nonrural.csv: 2590
Saved: /Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/grad2_nonrural.csv


In [24]:
#### SAVING A NEW CLASSIFICATION OF NON-RURAL IN NEW CSV

# If your final dataframe is named `result`
out_path = "/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/grad2_nonrural.csv"
result.to_csv(out_path, index=False)
print("Saved:", out_path)

Saved: /Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/grad2_nonrural.csv


In [25]:
#### GRADUATION RATES FOR RURAL2

import pandas as pd

# Load your file
df = pd.read_csv(
    "/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/grad2.csv",
    dtype=str
)
df.columns = df.columns.str.strip()

# Keep one cohort type (optional but recommended)
if "MEMBERSHIP_CODE" in df.columns:
    df = df[df["MEMBERSHIP_CODE"].astype(str).str.strip() == "11"].copy()

# Convert numeric fields (treat "-" as missing)
df["TOTAL_ENROLLED"] = pd.to_numeric(df["TOTAL_ENROLLED"], errors="coerce")
df["NUM_GRAD"] = pd.to_numeric(df["NUM_GRAD"], errors="coerce")

# Aggregate by subgroup
subgroup_grad = (
    df.groupby("SUBGROUP_NAME", as_index=False)
      .agg(
          TOTAL_ENROLLED=("TOTAL_ENROLLED", "sum"),
          NUM_GRAD=("NUM_GRAD", "sum")
      )
)

# Graduation rate = graduates / enrolled * 100
subgroup_grad["GRADUATION_RATE"] = (
    (subgroup_grad["NUM_GRAD"] / subgroup_grad["TOTAL_ENROLLED"]) * 100
).round(2)

# Optional: sort highest to lowest rate
subgroup_grad = subgroup_grad.sort_values("GRADUATION_RATE", ascending=False).reset_index(drop=True)

print(subgroup_grad)

                    SUBGROUP_NAME  TOTAL_ENROLLED  NUM_GRAD  GRADUATION_RATE
0  Never-English Language Learner           22010   19178.0            87.13
1                    All Students           22579   19650.0            87.03
2   Ever-English Language Learner             436     264.0            60.55
3    Non-English Language Learner           22446   13215.0            58.87
4        English Language Learner             133      21.0            15.79


In [26]:
#### COMPARING GRADUATION RATES FOR DIFFERENT CLASSIFICATIONS OF RURAL

import pandas as pd
from pathlib import Path

def subgroup_graduation_rates(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path, dtype=str)
    df.columns = df.columns.str.strip()

    # Optional cohort filter (recommended for consistency)
    if "MEMBERSHIP_CODE" in df.columns:
        df = df[df["MEMBERSHIP_CODE"].astype(str).str.strip() == "11"].copy()

    # Normalize numeric fields
    for col in ["TOTAL_ENROLLED", "NUM_GRAD"]:
        if col not in df.columns:
            raise ValueError(f"Missing required column '{col}' in {csv_path}")
        df[col] = df[col].replace("-", pd.NA)
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Keep valid denominator rows only
    df = df[df["TOTAL_ENROLLED"].notna() & (df["TOTAL_ENROLLED"] > 0)].copy()

    # Aggregate by subgroup
    out = (
        df.groupby("SUBGROUP_NAME", as_index=False)
          .agg(
              TOTAL_ENROLLED=("TOTAL_ENROLLED", "sum"),
              NUM_GRAD=("NUM_GRAD", "sum")
          )
    )

    # Graduation rate
    out["GRADUATION_RATE"] = ((out["NUM_GRAD"] / out["TOTAL_ENROLLED"]) * 100).round(2)
    out = out.sort_values("GRADUATION_RATE", ascending=False).reset_index(drop=True)
    return out

base = Path("/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis")

grad_rates = subgroup_graduation_rates(str(base / "grad.csv"))
grad2_rates = subgroup_graduation_rates(str(base / "grad2.csv"))

print("=== grad.csv ===")
display(grad_rates)

print("=== grad2.csv ===")
display(grad2_rates)

=== grad.csv ===


,SUBGROUP_NAME,TOTAL_ENROLLED,NUM_GRAD,GRADUATION_RATE
0,Never-English Language Learner,27800,23898.0,85.96
1,All Students,29811,25410.0,85.24
2,Ever-English Language Learner,1389,1124.0,80.92
3,Non-English Language Learner,29189,17867.0,61.21
4,English Language Learner,622,193.0,31.03


=== grad2.csv ===


,SUBGROUP_NAME,TOTAL_ENROLLED,NUM_GRAD,GRADUATION_RATE
0,Never-English Language Learner,22010,19178.0,87.13
1,All Students,22579,19650.0,87.03
2,Ever-English Language Learner,436,264.0,60.55
3,Non-English Language Learner,22446,13215.0,58.87
4,English Language Learner,133,21.0,15.79


In [31]:
#### COMPARING DROP-OUT RATES FOR DIFFERENT CLASSIFICATIONS OF RURAL

import pandas as pd
from pathlib import Path

def subgroup_dropout_rates(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path, dtype=str)
    df.columns = df.columns.str.strip()

    # Optional cohort filter (recommended for consistency)
    if "MEMBERSHIP_CODE" in df.columns:
        df = df[df["MEMBERSHIP_CODE"].astype(str).str.strip() == "11"].copy()

    # Normalize numeric fields
    for col in ["TOTAL_ENROLLED", "NUM_DROPOUT"]:
        if col not in df.columns:
            raise ValueError(f"Missing required column '{col}' in {csv_path}")
        df[col] = df[col].replace("-", pd.NA)
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Keep valid denominator rows only
    df = df[df["TOTAL_ENROLLED"].notna() & (df["TOTAL_ENROLLED"] > 0)].copy()

    # Aggregate by subgroup
    out = (
        df.groupby("SUBGROUP_NAME", as_index=False)
          .agg(
              TOTAL_ENROLLED=("TOTAL_ENROLLED", "sum"),
              NUM_DROPOUT=("NUM_DROPOUT", "sum")
          )
    )

    # Dropout rate
    out["DROPOUT_RATE"] = ((out["NUM_DROPOUT"] / out["TOTAL_ENROLLED"]) * 100).round(2)
    out = out.sort_values("DROPOUT_RATE", ascending=False).reset_index(drop=True)
    return out

base = Path("/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis")

grad_dropout_rates = subgroup_dropout_rates(str(base / "grad.csv"))
grad2_dropout_rates = subgroup_dropout_rates(str(base / "grad2.csv"))

print("=== grad.csv ===")
display(grad_dropout_rates)

print("=== grad2.csv ===")
display(grad2_dropout_rates)

=== grad.csv ===


,SUBGROUP_NAME,TOTAL_ENROLLED,NUM_DROPOUT,DROPOUT_RATE
0,English Language Learner,622,150.0,24.12
1,All Students,29811,2138.0,7.17
2,Never-English Language Learner,27800,1904.0,6.85
3,Non-English Language Learner,29189,1504.0,5.15
4,Ever-English Language Learner,1389,57.0,4.10


=== grad2.csv ===


,SUBGROUP_NAME,TOTAL_ENROLLED,NUM_DROPOUT,DROPOUT_RATE
0,English Language Learner,133,12.0,9.02
1,All Students,22579,1444.0,6.40
2,Never-English Language Learner,22010,1397.0,6.35
3,Non-English Language Learner,22446,1006.0,4.48
4,Ever-English Language Learner,436,12.0,2.75


In [32]:
#### #### COMPARING GRADUATION RATES FOR DIFFERENT CLASSIFICATIONS OF NON-RURAL

import pandas as pd
from pathlib import Path

def subgroup_graduation_rates(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path, dtype=str)
    df.columns = df.columns.str.strip()

    # Optional cohort filter (recommended for consistency)
    if "MEMBERSHIP_CODE" in df.columns:
        df = df[df["MEMBERSHIP_CODE"].astype(str).str.strip() == "11"].copy()

    # Normalize numeric fields
    for col in ["TOTAL_ENROLLED", "NUM_GRAD"]:
        if col not in df.columns:
            raise ValueError(f"Missing required column '{col}' in {csv_path}")
        df[col] = df[col].replace("-", pd.NA)
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Keep valid denominator rows only
    df = df[df["TOTAL_ENROLLED"].notna() & (df["TOTAL_ENROLLED"] > 0)].copy()

    # Aggregate by subgroup
    out = (
        df.groupby("SUBGROUP_NAME", as_index=False)
          .agg(
              TOTAL_ENROLLED=("TOTAL_ENROLLED", "sum"),
              NUM_GRAD=("NUM_GRAD", "sum")
          )
    )

    # Graduation rate
    out["GRADUATION_RATE"] = ((out["NUM_GRAD"] / out["TOTAL_ENROLLED"]) * 100).round(2)
    out = out.sort_values("GRADUATION_RATE", ascending=False).reset_index(drop=True)
    return out

base = Path("/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis")

grad_rates = subgroup_graduation_rates(str(base / "grad_nonrural.csv"))
grad2_rates = subgroup_graduation_rates(str(base / "grad2_nonrural.csv"))

print("=== grad_nonrural.csv ===")
display(grad_rates)

print("=== grad2_nonrural.csv ===")
display(grad2_rates)

=== grad_nonrural.csv ===


,SUBGROUP_NAME,TOTAL_ENROLLED,NUM_GRAD,GRADUATION_RATE
0,Ever-English Language Learner,27595,25126.0,91.05
1,Never-English Language Learner,130023,114950.0,88.41
2,All Students,169318,146378.0,86.45
3,Non-English Language Learner,157618,115725.0,73.42
4,English Language Learner,11700,5947.0,50.83


=== grad2_nonrural.csv ===


,SUBGROUP_NAME,TOTAL_ENROLLED,NUM_GRAD,GRADUATION_RATE
0,Ever-English Language Learner,28548,25986.0,91.03
1,Never-English Language Learner,135813,119670.0,88.11
2,All Students,176550,152138.0,86.17
3,Non-English Language Learner,164361,120377.0,73.24
4,English Language Learner,12189,6119.0,50.20


In [33]:
import pandas as pd
from pathlib import Path

def subgroup_dropout_rates(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path, dtype=str)
    df.columns = df.columns.str.strip()

    # Optional cohort filter (recommended for consistency)
    if "MEMBERSHIP_CODE" in df.columns:
        df = df[df["MEMBERSHIP_CODE"].astype(str).str.strip() == "11"].copy()

    # Normalize numeric fields
    for col in ["TOTAL_ENROLLED", "NUM_DROPOUT"]:
        if col not in df.columns:
            raise ValueError(f"Missing required column '{col}' in {csv_path}")
        df[col] = df[col].replace("-", pd.NA)
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Keep valid denominator rows only
    df = df[df["TOTAL_ENROLLED"].notna() & (df["TOTAL_ENROLLED"] > 0)].copy()

    # Aggregate by subgroup
    out = (
        df.groupby("SUBGROUP_NAME", as_index=False)
          .agg(
              TOTAL_ENROLLED=("TOTAL_ENROLLED", "sum"),
              NUM_DROPOUT=("NUM_DROPOUT", "sum")
          )
    )

    # Dropout rate
    out["DROPOUT_RATE"] = ((out["NUM_DROPOUT"] / out["TOTAL_ENROLLED"]) * 100).round(2)
    out = out.sort_values("DROPOUT_RATE", ascending=False).reset_index(drop=True)
    return out

base = Path("/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis")

grad_dropout_rates = subgroup_dropout_rates(str(base / "grad_nonrural.csv"))
grad2_dropout_rates = subgroup_dropout_rates(str(base / "grad2_nonrural.csv"))

print("=== grad_nonrural.csv ===")
display(grad_dropout_rates)

print("=== grad2_nonrural.csv ===")
display(grad2_dropout_rates)

=== grad_nonrural.csv ===


,SUBGROUP_NAME,TOTAL_ENROLLED,NUM_DROPOUT,DROPOUT_RATE
0,English Language Learner,11700,1706.0,14.58
1,All Students,169318,6855.0,4.05
2,Never-English Language Learner,130023,4528.0,3.48
3,Non-English Language Learner,157618,4427.0,2.81
4,Ever-English Language Learner,27595,585.0,2.12


=== grad2_nonrural.csv ===


,SUBGROUP_NAME,TOTAL_ENROLLED,NUM_DROPOUT,DROPOUT_RATE
0,English Language Learner,12189,1844.0,15.13
1,All Students,176550,7549.0,4.28
2,Never-English Language Learner,135813,5035.0,3.71
3,Non-English Language Learner,164361,4925.0,3.00
4,Ever-English Language Learner,28548,630.0,2.21


In [40]:
#### TAKING OUT ALL RURAL SCHOOLS in GRAD2.csv WITH NO ELL STUDENTS

import pandas as pd

in_path = "/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/grad2.csv"
out_path = "/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/grad2_ell_only.csv"

df = pd.read_csv(in_path, dtype=str)
df.columns = df.columns.str.strip()

# Normalize subgroup labels
sg = df["SUBGROUP_NAME"].fillna("").astype(str).str.strip().str.upper()

# Find schools with ELL subgroup and >0 enrolled ELL students
ell_rows = df[sg.isin(["ENGLISH LANGUAGE LEARNER", "ENGLISH LANGUAGE LEARNERS"])].copy()
ell_rows["TOTAL_ENROLLED_NUM"] = pd.to_numeric(
    ell_rows["TOTAL_ENROLLED"].replace("-", pd.NA), errors="coerce"
).fillna(0)

schools_with_ell = set(
    ell_rows.loc[ell_rows["TOTAL_ENROLLED_NUM"] > 0, "ENTITY_NAME"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# Keep only rows for those schools
df["ENTITY_NAME"] = df["ENTITY_NAME"].fillna("").astype(str).str.strip()
cleaned = df[df["ENTITY_NAME"].isin(schools_with_ell)].copy()

cleaned.to_csv(out_path, index=False)

print("Original rows:", len(df))
print("Cleaned rows:", len(cleaned))
print("Schools kept:", cleaned["ENTITY_NAME"].nunique())
print("Saved:", out_path)

Original rows: 1295
Cleaned rows: 315
Schools kept: 63
Saved: /Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/grad2_ell_only.csv


In [41]:
import pandas as pd

in_path = "/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/grad2_ell_only.csv"
out_path = "/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/grad_rate_per_school_with_ell.csv"

df = pd.read_csv(in_path, dtype=str)
df.columns = df.columns.str.strip()

# Keep the same cohort definition
df = df[df["MEMBERSHIP_CODE"].astype(str).str.strip() == "11"].copy()

# Numeric cleanup
for col in ["TOTAL_ENROLLED", "NUM_GRAD", "NUM_DROPOUT"]:
    df[col] = pd.to_numeric(df[col].replace("-", pd.NA), errors="coerce")

# Subsets
all_students = df[df["SUBGROUP_NAME"].astype(str).str.strip() == "All Students"].copy()
ell_students = df[df["SUBGROUP_NAME"].astype(str).str.strip() == "English Language Learner"].copy()

# Aggregate by school ID (safer than name-only)
all_agg = (
    all_students.groupby(["INSTITUTION_ID", "ENTITY_NAME"], as_index=False)
    .agg(
        total_students=("TOTAL_ENROLLED", "sum"),
        total_grad=("NUM_GRAD", "sum"),
        total_dropout=("NUM_DROPOUT", "sum"),
    )
)

ell_agg = (
    ell_students.groupby("INSTITUTION_ID", as_index=False)
    .agg(
        ell_grad=("NUM_GRAD", "sum"),
        ell_dropout=("NUM_DROPOUT", "sum"),
    )
)

# Merge
out = all_agg.merge(ell_agg, on="INSTITUTION_ID", how="left")

# Fill missing ELL counts with 0
out["ell_grad"] = out["ell_grad"].fillna(0)
out["ell_dropout"] = out["ell_dropout"].fillna(0)

# Avoid divide-by-zero
out = out[out["total_students"].notna() & (out["total_students"] > 0)].copy()

# Rates
out["grad_rate"] = (out["total_grad"] / out["total_students"] * 100).round(2)
out["dropout_rate"] = (out["total_dropout"] / out["total_students"] * 100).round(2)

# Per your request: ELL counts divided by total students (not ELL enrolled)
out["ell_grad_rate"] = (out["ell_grad"] / out["total_students"] * 100).round(2)
out["ell_dropout_rate"] = (out["ell_dropout"] / out["total_students"] * 100).round(2)

# Final 5 columns
final = out.rename(columns={"ENTITY_NAME": "entity_name"})[
    ["entity_name", "grad_rate", "dropout_rate", "ell_grad_rate", "ell_dropout_rate"]
]

final.to_csv(out_path, index=False)
print("Saved:", out_path)
final.head(20)

Saved: /Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/grad_rate_per_school_with_ell.csv


,entity_name,grad_rate,dropout_rate,ell_grad_rate,ell_dropout_rate
0,COOPERSTOWN CSD,89.66,5.17,0.00,0.00
1,HIGHLAND FALLS CSD,87.62,0.00,0.00,0.00
2,QUEENSBURY UFSD,93.39,3.72,0.00,0.00
3,ELLENVILLE CSD,76.52,9.85,1.52,0.76
4,WALLKILL CSD,90.88,5.61,0.00,0.00
5,SAUGERTIES CSD,89.02,5.49,1.22,1.83
6,ONTEORA CSD,83.00,9.00,0.00,0.00
7,NEW PALTZ CSD,94.97,0.50,0.00,0.00
8,LANSING CSD,92.16,1.96,0.00,0.00
9,GROTON CSD,87.50,1.56,0.00,0.00


In [42]:
import pandas as pd

in_path = "/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/grad2.csv"
out_path = "/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/grad_rate_per_school_fixed.csv"

df = pd.read_csv(in_path, dtype=str)
df.columns = df.columns.str.strip()

# Keep one cohort
df = df[df["MEMBERSHIP_CODE"].astype(str).str.strip() == "11"].copy()

# Normalize subgroup names
sg = df["SUBGROUP_NAME"].fillna("").astype(str).str.strip().str.upper()
df["SG_CLASS"] = pd.NA
df.loc[sg == "ALL STUDENTS", "SG_CLASS"] = "ALL"
df.loc[sg.isin(["ENGLISH LANGUAGE LEARNER", "ENGLISH LANGUAGE LEARNERS"]), "SG_CLASS"] = "ELL"
df = df[df["SG_CLASS"].notna()].copy()

# Numeric cleanup
df["TOTAL_ENROLLED"] = pd.to_numeric(df["TOTAL_ENROLLED"].replace("-", pd.NA), errors="coerce")
df["NUM_GRAD"] = pd.to_numeric(df["NUM_GRAD"].replace("-", pd.NA), errors="coerce").fillna(0)
df["NUM_DROPOUT"] = pd.to_numeric(df["NUM_DROPOUT"].replace("-", pd.NA), errors="coerce").fillna(0)

# Aggregate by school + subgroup
g = (
    df.groupby(["INSTITUTION_ID", "ENTITY_NAME", "SG_CLASS"], as_index=False)
      .agg(
          enrolled=("TOTAL_ENROLLED", "sum"),
          grad=("NUM_GRAD", "sum"),
          dropout=("NUM_DROPOUT", "sum")
      )
)

# Keep valid denominators
g = g[g["enrolled"].notna() & (g["enrolled"] > 0)].copy()

# Split
all_df = g[g["SG_CLASS"] == "ALL"][["INSTITUTION_ID", "ENTITY_NAME", "enrolled", "grad", "dropout"]].copy()
ell_df = g[g["SG_CLASS"] == "ELL"][["INSTITUTION_ID", "enrolled", "grad", "dropout"]].copy()

# Rename ELL columns
ell_df = ell_df.rename(columns={
    "enrolled": "ell_enrolled",
    "grad": "ell_grad",
    "dropout": "ell_dropout"
})

# Merge on ID
out = all_df.merge(ell_df, on="INSTITUTION_ID", how="left")

# Fill missing ELL values
out[["ell_enrolled", "ell_grad", "ell_dropout"]] = out[["ell_enrolled", "ell_grad", "ell_dropout"]].fillna(0)

# Rates (consistent denominators)
out["grad_rate"] = (out["grad"] / out["enrolled"] * 100).round(2)
out["dropout_rate"] = (out["dropout"] / out["enrolled"] * 100).round(2)
out["ell_grad_rate"] = (out["ell_grad"] / out["ell_enrolled"].where(out["ell_enrolled"] > 0) * 100).round(2)
out["ell_dropout_rate"] = (out["ell_dropout"] / out["ell_enrolled"].where(out["ell_enrolled"] > 0) * 100).round(2)

# Final 5 columns
final = out.rename(columns={"ENTITY_NAME": "entity_name"})[
    ["entity_name", "grad_rate", "dropout_rate", "ell_grad_rate", "ell_dropout_rate"]
]

final.to_csv(out_path, index=False)
print("Saved:", out_path)

# Quick sanity checks
print("Any rate > 100:",
      ((final[["grad_rate","dropout_rate","ell_grad_rate","ell_dropout_rate"]] > 100).any(axis=1)).sum())
final.head(20)

Saved: /Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/grad_rate_per_school_fixed.csv
Any rate > 100: 0


,entity_name,grad_rate,dropout_rate,ell_grad_rate,ell_dropout_rate
0,COOPERSTOWN CSD,89.66,5.17,0.0,0.0
1,HIGHLAND FALLS CSD,87.62,0.00,0.0,0.0
2,WHITEHALL CSD,85.71,6.12,NaN,NaN
3,CAMBRIDGE CSD,86.15,9.23,NaN,NaN
4,SALEM CSD,87.50,6.25,NaN,NaN
5,HUDSON FALLS CSD,78.65,9.55,NaN,NaN
6,HARTFORD CSD,96.77,3.23,NaN,NaN
7,GREENWICH CSD,96.67,0.00,NaN,NaN
8,FORT ANN CSD,67.86,10.71,NaN,NaN
9,FORT EDWARD UFSD,70.00,20.00,NaN,NaN


In [43]:
import pandas as pd
from pathlib import Path

# File names
files = ["grad2.csv", "grad2_nonrural.csv"]

ell_rows = []

for file in files:
    df = pd.read_csv(file, na_values=["-"])
    
    # Keep only English Language Learner rows
    ell = df[df["SUBGROUP_NAME"] == "English Language Learner"].copy()
    
    # Convert NUM_GRAD to numeric
    ell["NUM_GRAD"] = pd.to_numeric(ell["NUM_GRAD"], errors="coerce").fillna(0).astype(int)
    
    # Optional: track which file the row came from
    ell["SOURCE_FILE"] = Path(file).name
    
    ell_rows.append(ell)

# Combine both files into one table
ell_table = pd.concat(ell_rows, ignore_index=True)

# Total number of English Language Learner graduates across both files
total_ell_graduates = ell_table["NUM_GRAD"].sum()

print("Total ELL graduates across both files:", total_ell_graduates)

# Save the new table containing only ELL rows
ell_table.to_csv("english_language_learner_graduates_only.csv", index=False)

Total ELL graduates across both files: 6140


In [45]:
#### THIS IS THE ONLY ONE I THINK MAY BE CORRECT

import pandas as pd

def ell_dropout_average(csv_path):
    df = pd.read_csv(csv_path, dtype=str)
    df.columns = df.columns.str.strip()

    # Keep only English Language Learner rows
    ell = df[df["SUBGROUP_NAME"].astype(str).str.strip() == "English Language Learner"].copy()

    # Keep only rows with ELL students > 0
    ell["TOTAL_ENROLLED"] = pd.to_numeric(ell["TOTAL_ENROLLED"].replace("-", pd.NA), errors="coerce")
    ell = ell[ell["TOTAL_ENROLLED"] > 0].copy()

    # Convert dropout percentage to numeric
    ell["PER_DROPOUT"] = pd.to_numeric(ell["PER_DROPOUT"].replace("-", pd.NA), errors="coerce")

    avg_per_dropout = ell["PER_DROPOUT"].mean()

    return ell, avg_per_dropout

grad2_ell, grad2_avg_dropout = ell_dropout_average(
    "/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/grad2.csv"
)

grad2_nonrural_ell, grad2_nonrural_avg_dropout = ell_dropout_average(
    "/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/grad2_nonrural.csv"
)

print("grad2.csv average PER_DROPOUT:", grad2_avg_dropout)
display(grad2_ell)

print("grad2_nonrural.csv average PER_DROPOUT:", grad2_nonrural_avg_dropout)
display(grad2_nonrural_ell)

grad2.csv average PER_DROPOUT: 26.125


,SCHOOL_YEAR,INSTITUTION_ID,ENTITY_CD,ENTITY_NAME,MEMBERSHIP_CODE,MEMBERSHIP_DESC,SUBGROUP_NAME,TOTAL_ENROLLED,NUM_GRAD,PER_GRAD,...,PER_LOCAL_DIPLOMA,NUM_NON_DIPLOMA_CRED,PER_NON_DIPLOMA_CRED,NUM_STILL_ENROLLED,PER_STILL_ENROLLED,NUM_GED_TRANSFER,PER_GED_TRANSFER,NUM_DROPOUT,PER_DROPOUT,COUNTY
1,2023-24,800000055275,20101040000,ALFRED-ALMOND CSD,11,2020 Total Cohort - 4 Year Outcome - August 2024,English Language Learner,1,-,-,...,-,-,-,-,-,-,-,-,NaN,ALLEGANY
61,2023-24,800000055102,30101060000,CHENANGO FORKS CSD,11,2020 Total Cohort - 4 Year Outcome - August 2024,English Language Learner,1,-,-,...,-,-,-,-,-,-,-,-,NaN,BROOME
76,2023-24,800000054967,30701060000,CHENANGO VALLEY CSD,11,2020 Total Cohort - 4 Year Outcome - August 2024,English Language Learner,1,-,-,...,-,-,-,-,-,-,-,-,NaN,BROOME
96,2023-24,800000054912,31501060000,UNION-ENDICOTT CSD,11,2020 Total Cohort - 4 Year Outcome - August 2024,English Language Learner,11,2,18,...,0,0,0,8,73,0,0,1,9.0,BROOME
106,2023-24,800000054848,31701060000,WINDSOR CSD,11,2020 Total Cohort - 4 Year Outcome - August 2024,English Language Learner,1,-,-,...,-,-,-,-,-,-,-,-,NaN,BROOME
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1186,2023-24,800000036146,621201060000,ONTEORA CSD,11,2020 Total Cohort - 4 Year Outcome - August 2024,English Language Learner,4,-,-,...,-,-,-,-,-,-,-,-,NaN,ULSTER
1191,2023-24,800000036121,621601060000,SAUGERTIES CSD,11,2020 Total Cohort - 4 Year Outcome - August 2024,English Language Learner,7,2,29,...,14,0,0,2,29,0,0,3,43.0,ULSTER
1196,2023-24,800000036105,621801060000,WALLKILL CSD,11,2020 Total Cohort - 4 Year Outcome - August 2024,English Language Learner,1,-,-,...,-,-,-,-,-,-,-,-,NaN,ULSTER
1201,2023-24,800000036084,622002060000,ELLENVILLE CSD,11,2020 Total Cohort - 4 Year Outcome - August 2024,English Language Learner,5,2,40,...,0,0,0,2,40,0,0,1,20.0,ULSTER


grad2_nonrural.csv average PER_DROPOUT: 14.02030456852792


,SCHOOL_YEAR,INSTITUTION_ID,ENTITY_CD,ENTITY_NAME,MEMBERSHIP_CODE,MEMBERSHIP_DESC,SUBGROUP_NAME,TOTAL_ENROLLED,NUM_GRAD,PER_GRAD,...,NUM_LOCAL_DIPLOMA,PER_LOCAL_DIPLOMA,NUM_NON_DIPLOMA_CRED,PER_NON_DIPLOMA_CRED,NUM_STILL_ENROLLED,PER_STILL_ENROLLED,NUM_GED_TRANSFER,PER_GED_TRANSFER,NUM_DROPOUT,PER_DROPOUT
3,2023-24,800000055306,10802060000,GUILDERLAND CSD,11,2020 Total Cohort - 4 Year Outcome - August 2024,English Language Learner,2,-,-,...,-,-,-,-,-,-,-,-,-,NaN
13,2023-24,800000055260,11200010000,WATERVLIET CITY SD,11,2020 Total Cohort - 4 Year Outcome - August 2024,English Language Learner,3,-,-,...,-,-,-,-,-,-,-,-,-,NaN
17,2023-24,800000055729,10100010000,ALBANY CITY SD,11,2020 Total Cohort - 4 Year Outcome - August 2024,English Language Learner,76,39,51,...,6,8,1,1,27,36,0,0,9,12.0
27,2023-24,800000068133,10100860960,ALBANY LEADERSHIP CS-GIRLS,11,2020 Total Cohort - 4 Year Outcome - August 2024,English Language Learner,4,-,-,...,-,-,-,-,-,-,-,-,-,NaN
37,2023-24,800000055452,10306060000,BETHLEHEM CSD,11,2020 Total Cohort - 4 Year Outcome - August 2024,English Language Learner,1,-,-,...,-,-,-,-,-,-,-,-,-,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2566,2023-24,800000036680,590501060000,FALLSBURG CSD,11,2020 Total Cohort - 4 Year Outcome - August 2024,English Language Learner,15,3,20,...,0,0,0,0,7,47,0,0,5,33.0
2571,2023-24,800000036656,590901060000,LIBERTY CSD,11,2020 Total Cohort - 4 Year Outcome - August 2024,English Language Learner,13,5,38,...,1,8,0,0,3,23,0,0,5,38.0
2576,2023-24,800000036596,591401060000,MONTICELLO CSD,11,2020 Total Cohort - 4 Year Outcome - August 2024,English Language Learner,14,6,43,...,0,0,0,0,3,21,1,7,4,29.0
2581,2023-24,800000036448,610600010000,ITHACA CITY SD,11,2020 Total Cohort - 4 Year Outcome - August 2024,English Language Learner,5,2,40,...,0,0,1,20,1,20,0,0,1,20.0
